# 01 — Fine-tune YOLOv8 per Tennis (Player/Racket/Ball)

Esegui su Colab Pro con GPU (T4/A100).

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/smart-court-data'

In [ ]:
# Scarica dataset da Roboflow (inserisci API key)
from roboflow import Roboflow
rf = Roboflow(api_key='YOUR_API_KEY')
project = rf.workspace('tennis-ai').project('tennis-player-detection-4vnfq')
dataset = project.version(1).download('yolov8')
print(dataset.location)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    name='tennis_yolo',
    patience=20,
)
print('Training done. Best model:', results.save_dir)

In [ ]:
# Export ONNX
model.export(format='onnx', imgsz=640, simplify=True)
print('ONNX exported')

In [ ]:
# Scarica il modello su locale
from google.colab import files
files.download(f'{results.save_dir}/weights/best.pt')
files.download(f'{results.save_dir}/weights/best.onnx')